# Fog of War Transformer v2 — Multi-Modal Conditioning & Pretrained Vision

This notebook trains **FogOfWarTransformerV2**, a spatio-temporal architecture that predicts enemy positions through Fog of War using **hybrid conditioning**:

### Core Architecture Highlights:
1. **10 Champion Entity Stream (8 features per champion)**:
   - Kinematics: `x`, `y`, `confidence` (from YOLO)
   - Replay States: `is_dead`, `respawn_timer`, `level`, `creep_score`, `kda` (from `player_states.json`)
2. **Global Game Clock Token**:
   - Elapsed game seconds projected into all residual streams, capturing camp respawn windows and objective timings.
3. **Pretrained MobileNetV3 Visual Stream**:
   - Downsampled minimap screenshots (`64x64`) sampled at temporal stride (e.g., every 4s) into an environmental **Map Context Token**.
   - Preserves wave push lines, destroyed turrets, and vision illumination.
4. **Spatio-Temporal Transformer**:
   - **Temporal Attention**: Entities track their own kinematic momentum and respawn countdowns over time.
   - **Spatial Cross-Attention**: Champions attend to teammates, enemies, and the Map Context Token.
5. **5-Red Target Supervision (Dense Gradients)**:
   - Predicts Bivariate Gaussian distributions $(\mu_x, \mu_y, \sigma_x, \sigma_y, \rho)$ for **all 5 Red enemies** (Top, Jungle, Mid, Bot, Support).
   - Solves the sparse gradient bottleneck: visible laners anchor the map features, while the Jungler head piggybacks on the rich spatial representation.
6. **Loss Function**:
   - Bivariate Gaussian NLL + Auxiliary L1 Coordinate Penalty (anti-mean collapse) + FoW Importance Weighting (3x on FoW frames).


In [ ]:
# ── 1. Setup & Imports ──────────────────────────────────────────────────
import os
import time
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

# CUDA setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'Device name:  {torch.cuda.get_device_name(0)}')


In [ ]:
# ── 2. Hyperparameters & Configuration ────────────────────────────────
SEQ_LEN          = 60     # Temporal history length (60 frames / seconds)
VISUAL_STRIDE    = 4      # Sample 1 minimap image every 4 frames (15 visual tokens per window)
MAP_IMG_SIZE     = 64     # Downsampled minimap resolution (64x64 RGB)
USE_VISION       = True   # Set to False to run pure structured conditioning (instant test)

BATCH_SIZE       = 16     # Batch size
NUM_EPOCHS       = 35     # Training epochs
LEARNING_RATE    = 5e-4   # AdamW learning rate
WEIGHT_DECAY     = 1e-3   # Regularization
VAL_SPLIT        = 0.15   # 15% validation matches
SEED             = 42

# Model Dimensions
HIDDEN_DIM       = 256    # Transformer residual stream width
NUM_LAYERS       = 4      # Spatio-Temporal transformer layers
NHEAD            = 8      # Multi-head attention heads
DROPOUT          = 0.15   # Dropout rate
MAP_SIZE         = 512.0  # Pixel map coordinate space

# Loss Settings
FOW_LOSS_WEIGHT  = 3.0    # 3x gradient emphasis on frames where enemy is hidden in FoW
L1_LOSS_WEIGHT   = 5.0    # Auxiliary coordinate L1 penalty (prevents mean-guess collapse)

# Output Paths
OUTPUT_DIR = Path("models_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = OUTPUT_DIR / "fow_v2_best.pt"
LATEST_MODEL_PATH = OUTPUT_DIR / "fow_v2_latest.pt"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## 2. Multi-Modal Dataset Loading & Alignment

Each match in `Dataset/` contains:
- `Blue data/cleaned_match_data.csv`: Observer/Blue detections (`x, y, conf`)
- `All data/cleaned_match_data.csv`: Ground-truth positions for all champions
- `player_states.json`: Replay tabular states (`is_dead, respawn_timer, level, creep_score, kills, deaths, assists`)
- `roles.json`: Ground-truth champion-to-role mappings
- `0/Blue/*.png`: Minimap screenshots


In [ ]:
# ── 3. Multi-Modal Dataset Implementation ─────────────────────────────
# Automatically find Dataset directory relative to notebook or workspace
DATASET_ROOT = Path("Dataset") if Path("Dataset").exists() else Path("../Dataset")
print(f"Using Dataset path: {DATASET_ROOT.resolve()}")

ORDERED_ROLES = [
    'Blue_Top', 'Blue_Jungle', 'Blue_Mid', 'Blue_Bot', 'Blue_Support',
    'Red_Top', 'Red_Jungle', 'Red_Mid', 'Red_Bot', 'Red_Support'
]

class MatchSequenceData:
    # Holds parsed in-memory arrays for a single match.
    def __init__(self, match_id, entities_arr, targets_arr, game_times, screenshot_paths):
        self.match_id = match_id
        self.entities = entities_arr        # (N_frames, 10, 8) float32
        self.targets = targets_arr          # (N_frames, 5, 2)  float32 (Red team true x, y in [0, 1])
        self.game_times = game_times        # (N_frames, 1)     float32
        self.screenshot_paths = screenshot_paths # List of Path objects (or None)
        self.n_frames = len(game_times)

def load_match(match_folder: Path):
    blue_csv = match_folder / "Blue data" / "cleaned_match_data.csv"
    all_csv = match_folder / "All data" / "cleaned_match_data.csv"
    states_file = match_folder / "player_states.json"
    roles_file = match_folder / "roles.json"
    shots_dir = match_folder / "0" / "Blue"

    if not (blue_csv.exists() and all_csv.exists() and states_file.exists() and roles_file.exists()):
        return None

    try:
        df_in = pd.read_csv(blue_csv)
        df_targ = pd.read_csv(all_csv)
        with open(states_file, "r", encoding="utf-8") as f:
            states = json.load(f)
        with open(roles_file, "r", encoding="utf-8") as f:
            roles = json.load(f)
    except Exception:
        return None

    role_to_champ = {v: k for k, v in roles.items()}
    blue_frames = set(df_in['frame'].unique())
    all_frames = set(df_targ['frame'].unique())
    state_frames = set(int(k) for k in states.keys())
    common = sorted(list(blue_frames.intersection(all_frames).intersection(state_frames)))

    if len(common) <= SEQ_LEN:
        return None

    # Pivot kinematics
    p_in = df_in.pivot(index='frame', columns='role', values=['x', 'y', 'confidence']).reindex(
        columns=pd.MultiIndex.from_product([['x', 'y', 'confidence'], ORDERED_ROLES])
    ).loc[common]

    p_targ = df_targ.pivot(index='frame', columns='role', values=['x', 'y']).reindex(
        columns=pd.MultiIndex.from_product([['x', 'y'], ORDERED_ROLES[5:]]) # 5 Red roles
    ).loc[common]

    n_f = len(common)
    entities = np.zeros((n_f, 10, 8), dtype=np.float32)

    # 1. Kinematics (x, y, conf)
    for r_idx, r_name in enumerate(ORDERED_ROLES):
        x_vals = p_in['x'][r_name].values
        y_vals = p_in['y'][r_name].values
        conf_vals = p_in['confidence'][r_name].values

        # Normalize x, y in [0, 1] (-1 stays -1)
        entities[:, r_idx, 0] = np.where(x_vals >= 0, x_vals / MAP_SIZE, x_vals)
        entities[:, r_idx, 1] = np.where(y_vals >= 0, y_vals / MAP_SIZE, y_vals)
        entities[:, r_idx, 2] = np.clip(conf_vals, 0.0, 1.0)

    # 2. Metadata (is_dead, respawn_timer, level, cs, kda)
    for f_idx, f_num in enumerate(common):
        f_str = str(f_num)
        f_state = states.get(f_str, {})
        for r_idx, r_name in enumerate(ORDERED_ROLES):
            c_name = role_to_champ.get(r_name)
            if c_name and c_name in f_state:
                c_info = f_state[c_name]
                entities[f_idx, r_idx, 3] = 1.0 if c_info.get('is_dead') else 0.0
                entities[f_idx, r_idx, 4] = float(c_info.get('respawn_timer', 0.0)) / 70.0
                entities[f_idx, r_idx, 5] = float(c_info.get('level', 1)) / 18.0
                entities[f_idx, r_idx, 6] = float(c_info.get('creep_score', 0)) / 300.0
                kda = float(c_info.get('kills', 0) + c_info.get('assists', 0)) / max(1.0, float(c_info.get('deaths', 1)))
                entities[f_idx, r_idx, 7] = min(kda / 10.0, 1.0)

    # 3. Targets (Red team 5 roles true coordinates)
    targets = np.zeros((n_f, 5, 2), dtype=np.float32)
    for r_idx, r_name in enumerate(ORDERED_ROLES[5:]):
        tx = p_targ['x'][r_name].values
        ty = p_targ['y'][r_name].values
        targets[:, r_idx, 0] = np.where(tx >= 0, tx / MAP_SIZE, tx)
        targets[:, r_idx, 1] = np.where(ty >= 0, ty / MAP_SIZE, ty)

    # 4. Game clock
    game_times = (np.array(common, dtype=np.float32) / 1800.0).reshape(-1, 1)

    # 5. Screenshot file paths
    shot_paths = [shots_dir / f"{f_num}.png" for f_num in common]

    return MatchSequenceData(match_folder.name, entities, targets, game_times, shot_paths)

print("Discovering and indexing cleaned matches...")
all_matches = sorted([m for m in DATASET_ROOT.iterdir() if m.is_dir() and not m.name.startswith('_')])
parsed_matches = []
for m_dir in tqdm(all_matches, desc="Parsing Matches"):
    m_data = load_match(m_dir)
    if m_data is not None:
        parsed_matches.append(m_data)

print(f"Successfully parsed {len(parsed_matches)} complete matches!")

# Match-level split
random.seed(SEED)
random.shuffle(parsed_matches)
n_val = max(1, int(len(parsed_matches) * VAL_SPLIT))
val_matches = parsed_matches[:n_val]
train_matches = parsed_matches[n_val:]
print(f"Train Matches: {len(train_matches)} | Validation Matches: {len(val_matches)}")


class LeagueMultiModalDataset(Dataset):
    # Sliding window dataset over matches.
    def __init__(self, matches, seq_len=60, visual_stride=4, img_size=64, use_vision=True):
        self.matches = matches
        self.seq_len = seq_len
        self.visual_stride = visual_stride
        self.img_size = img_size
        self.use_vision = use_vision
        self.samples = []

        for m_idx, m in enumerate(matches):
            n_windows = m.n_frames - seq_len
            for w_start in range(n_windows):
                self.samples.append((m_idx, w_start))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        m_idx, start = self.samples[idx]
        m = self.matches[m_idx]
        end = start + self.seq_len

        # Slices
        entities_slice = torch.tensor(m.entities[start:end], dtype=torch.float32)  # (T, 10, 8)
        game_time_slice = torch.tensor(m.game_times[start:end], dtype=torch.float32) # (T, 1)
        target = torch.tensor(m.targets[end - 1], dtype=torch.float32)             # (5, 2)
        fow_mask = (entities_slice[-1, 5:, 2] <= 0.0)                              # (5,) Red team in FoW at t=T-1

        # Minimap images (sampled at visual_stride)
        if self.use_vision:
            stride_indices = list(range(start, end, self.visual_stride))
            imgs = []
            for s_i in stride_indices:
                p = m.screenshot_paths[s_i]
                if not p.exists() and p.with_suffix('.jpg').exists():
                    p = p.with_suffix('.jpg')
                if p.exists():
                    im = cv2.imread(str(p))
                    if im is not None:
                        im = cv2.resize(im, (self.img_size, self.img_size))
                        im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
                        im = np.transpose(im, (2, 0, 1)) # (3, H, W)
                        imgs.append(im)
                    else:
                        imgs.append(np.zeros((3, self.img_size, self.img_size), dtype=np.float32))
                else:
                    imgs.append(np.zeros((3, self.img_size, self.img_size), dtype=np.float32))
            map_tensor = torch.tensor(np.stack(imgs, axis=0), dtype=torch.float32) # (T_vis, 3, H, W)
        else:
            map_tensor = torch.zeros(1, dtype=torch.float32)

        return entities_slice, game_time_slice, map_tensor, target, fow_mask

train_dataset = LeagueMultiModalDataset(train_matches, SEQ_LEN, VISUAL_STRIDE, MAP_IMG_SIZE, USE_VISION)
val_dataset   = LeagueMultiModalDataset(val_matches, SEQ_LEN, VISUAL_STRIDE, MAP_IMG_SIZE, USE_VISION)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"Total Sliding Windows -> Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")


## 3. Model Architecture & Loss Function

The architecture combines:
1. `MinimapVisualEncoder`: Pretrained `MobileNetV3-Small` backbone with frozen edge-detection layers.
2. `FogOfWarTransformerV2`: Spatio-temporal transformer operating on 11 entities (1 Map Context + 10 Champions).
3. `bivariate_gaussian_fow_loss`: Joint NLL + Auxiliary L1 penalty + FoW weighting on all 5 Red enemies.


In [ ]:
# ── 4. FogOfWarTransformerV2 Architecture & Loss ─────────────────────
class MinimapVisualEncoder(nn.Module):
    def __init__(self, embed_dim=256, pretrained=True):
        super().__init__()
        try:
            weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
            base = models.mobilenet_v3_small(weights=weights)
        except Exception:
            base = models.mobilenet_v3_small(weights=None)

        self.features = base.features
        # Freeze early low-level filters (edges/colors)
        for param in self.features[:4].parameters():
            param.requires_grad = False

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Linear(576, embed_dim)

    def forward(self, x):
        # x: (B, T_vis, 3, H, W)
        B, T_vis, C, H, W = x.shape
        x_flat = x.view(B * T_vis, C, H, W)
        feat = self.features(x_flat)
        feat = self.pool(feat).flatten(1)
        proj = self.proj(feat)
        return proj.view(B, T_vis, -1) # (B, T_vis, embed_dim)


class FogOfWarTransformerV2(nn.Module):
    def __init__(
        self,
        num_roles=10,
        num_predict_roles=5,
        entity_feat_dim=8,
        hidden_dim=256,
        nhead=8,
        num_layers=4,
        seq_len=60,
        visual_stride=4,
        dropout=0.15,
        use_vision=True
    ):
        super().__init__()
        self.num_roles = num_roles
        self.num_predict_roles = num_predict_roles
        self.seq_len = seq_len
        self.visual_stride = visual_stride
        self.hidden_dim = hidden_dim
        self.use_vision = use_vision

        # 1. Champion Entity Projection
        self.champ_proj = nn.Linear(entity_feat_dim, hidden_dim)

        # 2. Learnable Role Embeddings (10 roles)
        self.role_embed = nn.Parameter(torch.randn(1, 1, num_roles, hidden_dim) * 0.02)

        # 3. Learnable Temporal Positional Embeddings
        self.time_embed = nn.Parameter(torch.randn(1, seq_len, 1, hidden_dim) * 0.02)

        # 4. Global Game Clock MLP
        self.clock_mlp = nn.Sequential(
            nn.Linear(1, 64),
            nn.GELU(),
            nn.Linear(64, hidden_dim)
        )

        # 5. Visual Minimap Encoder (MobileNetV3)
        if use_vision:
            self.visual_encoder = MinimapVisualEncoder(embed_dim=hidden_dim, pretrained=True)
            self.map_role_embed = nn.Parameter(torch.randn(1, 1, 1, hidden_dim) * 0.02)
        else:
            self.visual_encoder = None

        # 6. Spatio-Temporal Transformer Layers
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'time_norm': nn.LayerNorm(hidden_dim),
                'time_attn': nn.MultiheadAttention(hidden_dim, nhead, dropout=dropout, batch_first=True),
                'spatial_norm': nn.LayerNorm(hidden_dim),
                'spatial_attn': nn.MultiheadAttention(hidden_dim, nhead, dropout=dropout, batch_first=True),
                'ffn_norm': nn.LayerNorm(hidden_dim),
                'ffn': nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim * 4),
                    nn.GELU(),
                    nn.Dropout(dropout),
                    nn.Linear(hidden_dim * 4, hidden_dim),
                    nn.Dropout(dropout)
                )
            }) for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_dim)

        # 7. Output Head: Maps the 5 Red enemy residual tokens to 5 Gaussian params
        self.output_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 5)
        )

    def forward(self, x_entities, game_time, map_images=None):
        B, T, R, _ = x_entities.shape

        # Champions stream
        h_champs = (
            self.champ_proj(x_entities)
            + self.role_embed
            + self.time_embed[:, :T, :, :]
            + self.clock_mlp(game_time).unsqueeze(2)
        ) # (B, T, 10, hidden_dim)

        if self.use_vision and map_images is not None and map_images.dim() == 5:
            # Visual feature map: (B, T_vis, hidden_dim)
            v_feats = self.visual_encoder(map_images)
            # Temporal interpolation across T frames
            v_feats_t = F.interpolate(
                v_feats.permute(0, 2, 1),
                size=T,
                mode='linear',
                align_corners=False
            ).permute(0, 2, 1) # (B, T, hidden_dim)

            map_token = (v_feats_t.unsqueeze(2) + self.map_role_embed + self.time_embed[:, :T, :, :])
            h = torch.cat([map_token, h_champs], dim=2) # (B, T, 11, hidden_dim)
            red_start_idx = 6 # 0=Map, 1-5=Blue, 6-10=Red
        else:
            h = h_champs
            red_start_idx = 5 # 0-4=Blue, 5-9=Red

        N_tokens = h.shape[2]

        # Spatio-Temporal Attention Highway
        for layer in self.layers:
            # 1. Temporal Attention (along time)
            h_time_in = h.permute(0, 2, 1, 3).reshape(B * N_tokens, T, self.hidden_dim)
            h_tnorm = layer['time_norm'](h_time_in)
            h_tattn, _ = layer['time_attn'](h_tnorm, h_tnorm, h_tnorm)
            h = (h_time_in + h_tattn).reshape(B, N_tokens, T, self.hidden_dim).permute(0, 2, 1, 3)

            # 2. Spatial Attention (across entities + map)
            h_spat_in = h.reshape(B * T, N_tokens, self.hidden_dim)
            h_snorm = layer['spatial_norm'](h_spat_in)
            h_sattn, _ = layer['spatial_attn'](h_snorm, h_snorm, h_snorm)
            h = (h_spat_in + h_sattn).reshape(B, T, N_tokens, self.hidden_dim)

            # 3. FFN
            h_fnorm = layer['ffn_norm'](h)
            h = h + layer['ffn'](h_fnorm)

        h = self.final_norm(h)

        # Readout from the latest frame (T - 1) for the 5 Red enemies
        last_red_states = h[:, -1, red_start_idx : red_start_idx + self.num_predict_roles, :] # (B, 5, hidden_dim)
        out = self.output_head(last_red_states) # (B, 5, 5)

        mu_x  = torch.sigmoid(out[..., 0])
        mu_y  = torch.sigmoid(out[..., 1])
        sig_x = F.softplus(out[..., 2]) + 1e-4
        sig_y = F.softplus(out[..., 3]) + 1e-4
        rho   = torch.tanh(out[..., 4])

        return torch.stack([mu_x, mu_y, sig_x, sig_y, rho], dim=-1) # (B, 5, 5)


def bivariate_gaussian_fow_loss(predictions, targets, fow_mask=None, mask=None, fow_weight=3.0, l1_weight=5.0):
    # Combined Bivariate Gaussian NLL + Auxiliary L1 Coordinate Penalty + FoW Importance Weighting.
    # predictions: (B, 5, 5) -> [mu_x, mu_y, sigma_x, sigma_y, rho] for Red team
    # targets:     (B, 5, 2) -> [x_gt, y_gt] in [0, 1] range (-1 = dead/missing)
    # fow_mask:    (B, 5) boolean tensor, True if enemy was unseen in FoW at t=T-1
    # mask:        (B, 5) boolean tensor, True for valid targets
    mu_x    = predictions[..., 0]
    mu_y    = predictions[..., 1]
    sigma_x = torch.clamp(predictions[..., 2], min=1e-4, max=10.0)
    sigma_y = torch.clamp(predictions[..., 3], min=1e-4, max=10.0)
    rho     = torch.clamp(predictions[..., 4], min=-0.999, max=0.999)

    x_gt = targets[..., 0]
    y_gt = targets[..., 1]

    z_x = (x_gt - mu_x) / sigma_x
    z_y = (y_gt - mu_y) / sigma_y

    one_minus_rho2 = 1.0 - (rho ** 2)
    Z = (z_x ** 2 + z_y ** 2 - 2 * rho * z_x * z_y) / one_minus_rho2
    nll = torch.log(sigma_x) + torch.log(sigma_y) + 0.5 * torch.log(one_minus_rho2) + 0.5 * Z

    # Auxiliary L1 coordinate loss
    l1_dist = torch.abs(x_gt - mu_x) + torch.abs(y_gt - mu_y)
    total_loss = nll + l1_weight * l1_dist

    if fow_mask is not None:
        fow_mult = torch.where(fow_mask, fow_weight, 1.0)
        total_loss = total_loss * fow_mult

    if mask is not None:
        if mask.sum() == 0:
            return torch.tensor(0.0, device=predictions.device, requires_grad=True)
        total_loss = total_loss[mask]

    return total_loss.mean()


## 4. Model Initialization & Dry-Run Sanity Check


In [ ]:
# ── 5. Initialize Model, Optimizer, & LR Scheduler ───────────────────
model = FogOfWarTransformerV2(
    num_roles=10,
    num_predict_roles=5,
    entity_feat_dim=8,
    hidden_dim=HIDDEN_DIM,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    seq_len=SEQ_LEN,
    visual_stride=VISUAL_STRIDE,
    dropout=DROPOUT,
    use_vision=USE_VISION
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Initialized FogOfWarTransformerV2 | Learnable parameters: {total_params:,}')

# Dry run with sample batch
s_ent, s_time, s_map, s_targ, s_fow = next(iter(train_loader))
s_ent, s_time, s_map = s_ent.to(device), s_time.to(device), s_map.to(device)
s_targ, s_fow = s_targ.to(device), s_fow.to(device)

model.eval()
with torch.no_grad():
    s_pred = model(s_ent, s_time, s_map if USE_VISION else None)
    s_valid = (s_targ[:, :, 0] >= 0)
    s_loss = bivariate_gaussian_fow_loss(s_pred, s_targ, fow_mask=s_fow, mask=s_valid)

print(f'[OK] Entities shape:     {s_ent.shape}   (Expected: [B, {SEQ_LEN}, 10, 8])')
print(f'[OK] Clock shape:        {s_time.shape}  (Expected: [B, {SEQ_LEN}, 1])')
if USE_VISION:
    print(f'[OK] Map images shape:   {s_map.shape}   (Expected: [B, {SEQ_LEN // VISUAL_STRIDE}, 3, {MAP_IMG_SIZE}, {MAP_IMG_SIZE}])')
print(f'[OK] Target shape:       {s_targ.shape}  (Expected: [B, 5, 2] Red Team)')
print(f'[OK] Prediction shape:   {s_pred.shape}  (Expected: [B, 5, 5] Red Team)')
print(f'[OK] Initial dry loss:   {s_loss.item():.4f}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)


## 5. Training Loop with Live Metrics & Checkpointing


In [ ]:
# ── 6. Full Training Loop ─────────────────────────────────────────────
history = {
    'train_loss': [], 'train_sigma': [],
    'val_loss': [], 'val_sigma': [], 'val_jg_error_px': [],
    'lr': []
}
best_val_loss = float('inf')
start_time = time.time()

print('=' * 95)
print(f"{'Epoch':<8} | {'Train Loss':<12} | {'Train Sigma (px)':<17} | {'Val Loss':<12} | {'Val Sigma (px)':<15} | {'Jg Err (px)':<12} | {'LR':<9}")
print('=' * 95)

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train Step ──────────────────────────────────────────────
    model.train()
    running_loss, running_sigma, total_train_batches = 0.0, 0.0, 0

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS} [Train]", leave=False, unit="batch")
    for b_ent, b_time, b_map, b_targ, b_fow in train_pbar:
        b_ent = b_ent.to(device)
        b_time = b_time.to(device)
        b_map = b_map.to(device) if USE_VISION else None
        b_targ = b_targ.to(device)
        b_fow = b_fow.to(device)

        valid_mask = (b_targ[:, :, 0] >= 0)

        optimizer.zero_grad()
        pred = model(b_ent, b_time, b_map)
        loss = bivariate_gaussian_fow_loss(
            pred, b_targ,
            fow_mask=b_fow,
            mask=valid_mask,
            fow_weight=FOW_LOSS_WEIGHT,
            l1_weight=L1_LOSS_WEIGHT
        )
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_loss = loss.item()
        batch_sigma = pred[:, :, 2:4].mean().item() * MAP_SIZE

        running_loss += batch_loss
        running_sigma += batch_sigma
        total_train_batches += 1

        train_pbar.set_postfix({'loss': f'{batch_loss:.3f}', 'sigma_px': f'{batch_sigma:.1f}'})

    train_loss = running_loss / total_train_batches
    train_sigma = running_sigma / total_train_batches

    # ── Validation Step ─────────────────────────────────────────
    model.eval()
    val_running_loss, val_running_sigma, total_val_batches = 0.0, 0.0, 0
    val_jg_errors = []

    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS} [Val]", leave=False, unit="batch")
    with torch.no_grad():
        for b_ent, b_time, b_map, b_targ, b_fow in val_pbar:
            b_ent = b_ent.to(device)
            b_time = b_time.to(device)
            b_map = b_map.to(device) if USE_VISION else None
            b_targ = b_targ.to(device)
            b_fow = b_fow.to(device)

            valid_mask = (b_targ[:, :, 0] >= 0)
            pred = model(b_ent, b_time, b_map)
            loss = bivariate_gaussian_fow_loss(
                pred, b_targ,
                fow_mask=b_fow,
                mask=valid_mask,
                fow_weight=FOW_LOSS_WEIGHT,
                l1_weight=L1_LOSS_WEIGHT
            )

            val_running_loss += loss.item()
            val_running_sigma += pred[:, :, 2:4].mean().item() * MAP_SIZE
            total_val_batches += 1

            # Track Jungler Euclidean error (role index 1 = Red_Jungle)
            jg_valid = (b_targ[:, 1, 0] >= 0)
            if jg_valid.sum() > 0:
                pred_jg_xy = pred[jg_valid, 1, 0:2] * MAP_SIZE
                targ_jg_xy = b_targ[jg_valid, 1, 0:2] * MAP_SIZE
                dist = torch.norm(pred_jg_xy - targ_jg_xy, dim=-1).cpu().numpy()
                val_jg_errors.extend(dist.tolist())

    val_loss = val_running_loss / total_val_batches
    val_sigma = val_running_sigma / total_val_batches
    val_jg_err = np.mean(val_jg_errors) if val_jg_errors else 0.0

    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_sigma'].append(train_sigma)
    history['val_loss'].append(val_loss)
    history['val_sigma'].append(val_sigma)
    history['val_jg_error_px'].append(val_jg_err)
    history['lr'].append(current_lr)

    star = ' '
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), str(BEST_MODEL_PATH))
        star = ' * (best)'

    torch.save(model.state_dict(), str(LATEST_MODEL_PATH))
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS:02d} | {train_loss:<12.4f} | {train_sigma:<17.2f} | {val_loss:<12.4f} | {val_sigma:<15.2f} | {val_jg_err:<12.2f} | {current_lr:<9.5f}{star}")

elapsed = time.time() - start_time
print('=' * 95)
print(f'Training complete in {elapsed / 60:.2f} mins! Best Val Loss: {best_val_loss:.4f}')
print(f'Saved checkpoints to: {BEST_MODEL_PATH}')


## 6. Minimap Prediction & Uncertainty Visualizer

Renders 2-sigma Gaussian ellipses for all 5 Red enemies over actual minimap screenshots, with special prominent styling for the **Red Jungler**.


In [ ]:
# ── 7. Minimap Prediction Visualizer ──────────────────────────────────
# Select a random unseen validation match
selected_match = random.choice(val_matches)
match_id = selected_match.match_id
print(f"Visualizing Predictions on Match: {match_id}")

if BEST_MODEL_PATH.exists():
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    print(f"Loaded best checkpoint from {BEST_MODEL_PATH}")
model.eval()

RED_ROLE_NAMES = ['Red_Top', 'Red_Jungle', 'Red_Mid', 'Red_Bot', 'Red_Support']
RED_ROLE_TAGS  = ['R-Top', 'R-Jungle', 'R-Mid', 'R-Bot', 'R-Sup']
BLUE_ROLE_TAGS = ['B-Top', 'B-Jungle', 'B-Mid', 'B-Bot', 'B-Sup']

def render_prediction_frame(window_idx, n_std=2.0, show_gt=True):
    start = window_idx
    end = start + SEQ_LEN
    target_f_idx = end - 1

    img_path = selected_match.screenshot_paths[target_f_idx]
    if not img_path.exists() and img_path.with_suffix('.jpg').exists():
        img_path = img_path.with_suffix('.jpg')
    if img_path.exists():
        frame = cv2.imread(str(img_path))
        if frame.shape[:2] != (512, 512):
            frame = cv2.resize(frame, (512, 512))
    else:
        frame = np.full((512, 512, 3), 35, dtype=np.uint8)

    overlay = frame.copy()

    # Prepare inputs
    x_ent = torch.tensor(selected_match.entities[start:end], dtype=torch.float32).unsqueeze(0).to(device)
    x_time = torch.tensor(selected_match.game_times[start:end], dtype=torch.float32).unsqueeze(0).to(device)

    if USE_VISION:
        stride_indices = list(range(start, end, VISUAL_STRIDE))
        imgs = []
        for s_i in stride_indices:
            p = selected_match.screenshot_paths[s_i]
            if p.exists():
                im = cv2.imread(str(p))
                if im is not None:
                    im = cv2.resize(im, (MAP_IMG_SIZE, MAP_IMG_SIZE))
                    im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
                    im = np.transpose(im, (2, 0, 1))
                    imgs.append(im)
                else:
                    imgs.append(np.zeros((3, MAP_IMG_SIZE, MAP_IMG_SIZE), dtype=np.float32))
            else:
                imgs.append(np.zeros((3, MAP_IMG_SIZE, MAP_IMG_SIZE), dtype=np.float32))
        x_map = torch.tensor(np.stack(imgs, axis=0), dtype=torch.float32).unsqueeze(0).to(device)
    else:
        x_map = None

    with torch.no_grad():
        preds = model(x_ent, x_time, x_map)[0].cpu().numpy() # (5, 5)

    gt = selected_match.targets[target_f_idx] # (5, 2)

    # 1. Draw 2-Sigma Gaussian Uncertainty Ellipses
    for r_idx in range(5):
        mu_x = preds[r_idx, 0] * MAP_SIZE
        mu_y = preds[r_idx, 1] * MAP_SIZE
        sig_x = preds[r_idx, 2] * MAP_SIZE
        sig_y = preds[r_idx, 3] * MAP_SIZE
        rho = preds[r_idx, 4]

        cov = np.array([[sig_x**2, rho * sig_x * sig_y], [rho * sig_x * sig_y, sig_y**2]], dtype=np.float32)
        eigenvals, eigenvecs = np.linalg.eigh(cov)
        order = eigenvals.argsort()[::-1]
        eigenvals, eigenvecs = eigenvals[order], eigenvecs[:, order]

        axis_major = max(int(n_std * np.sqrt(max(eigenvals[0], 1e-4))), 4)
        axis_minor = max(int(n_std * np.sqrt(max(eigenvals[1], 1e-4))), 4)
        angle = np.degrees(np.arctan2(eigenvecs[1, 0], eigenvecs[0, 0]))

        # Red Jungler gets bright orange/gold tint; others get red tint
        tint = (30, 160, 255) if r_idx == 1 else (40, 50, 230)
        cv2.ellipse(
            overlay,
            (int(np.clip(mu_x, 0, 511)), int(np.clip(mu_y, 0, 511))),
            (axis_major, axis_minor),
            angle, 0, 360,
            tint,
            -1
        )

    cv2.addWeighted(overlay, 0.40, frame, 0.60, 0, frame)

    # 2. Known Blue Team Positions from t = T-1
    for b_idx in range(5):
        bx = selected_match.entities[end - 1, b_idx, 0]
        by = selected_match.entities[end - 1, b_idx, 1]
        if bx >= 0 and by >= 0:
            b_px, b_py = int(bx * MAP_SIZE), int(by * MAP_SIZE)
            cv2.circle(frame, (b_px, b_py), 4, (255, 200, 50), -1)
            cv2.putText(frame, BLUE_ROLE_TAGS[b_idx], (b_px + 6, b_py + 4), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 200, 50), 1)

    # 3. Red Team Predictions, Ground Truth Crosshairs, and Error Lines
    jg_err = 0.0
    for r_idx in range(5):
        tag = RED_ROLE_TAGS[r_idx]
        mu_x = int(np.clip(preds[r_idx, 0] * MAP_SIZE, 0, 511))
        mu_y = int(np.clip(preds[r_idx, 1] * MAP_SIZE, 0, 511))

        # Ground Truth Marker (Lime Green Crosshair)
        if show_gt and gt[r_idx, 0] >= 0:
            gt_x = int(gt[r_idx, 0] * MAP_SIZE)
            gt_y = int(gt[r_idx, 1] * MAP_SIZE)
            cv2.drawMarker(frame, (gt_x, gt_y), (50, 255, 120), markerType=cv2.MARKER_CROSS, markerSize=10, thickness=2)

            dist = np.hypot(mu_x - gt_x, mu_y - gt_y)
            if r_idx == 1:
                jg_err = dist
            cv2.line(frame, (mu_x, mu_y), (gt_x, gt_y), (180, 180, 180), 1, cv2.LINE_AA)

        # Center Prediction Dot
        dot_color = (0, 180, 255) if r_idx == 1 else (50, 60, 240)
        cv2.circle(frame, (mu_x, mu_y), 5, (0, 0, 0), -1)
        cv2.circle(frame, (mu_x, mu_y), 4, dot_color, -1)
        cv2.putText(frame, tag, (mu_x + 6, mu_y + 4), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 0, 0), 2)
        cv2.putText(frame, tag, (mu_x + 6, mu_y + 4), cv2.FONT_HERSHEY_SIMPLEX, 0.38, dot_color, 1)

    return frame, jg_err

# Plot inline 4 multi-frame snapshots
total_testable = selected_match.n_frames - SEQ_LEN
sample_indices = [
    int(total_testable * 0.15),
    int(total_testable * 0.40),
    int(total_testable * 0.65),
    int(total_testable * 0.90)
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
for ax, s_idx in zip(axes, sample_indices):
    rendered_bgr, jg_err = render_prediction_frame(s_idx, n_std=2.0, show_gt=True)
    rendered_rgb = cv2.cvtColor(rendered_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(rendered_rgb)
    ax.set_title(f"Window {s_idx} | Jungler Error: {jg_err:.1f}px", fontsize=11)
    ax.axis("off")

plt.suptitle(
    f"FogOfWarTransformerV2 Predictions on Validation Match: {match_id}\n" +
    "[Gold/Orange: Predicted Jungler 2-Sigma Heatmap]  |  [Red: Other 4 Enemies]  |  [Green +: Ground Truth]",
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()
